# EGC KEGG

Builds the KEGG annotation list from `kegg_db.json` and writes one `.ko.json`
per sample to `DataStructures/Kegg/`.

KEGG is a flat list of KOs, exactly as JGI assigned them: every KO is a
depth-1 entry with no parent and no category, named from `ko_names.json`
(JGI's product names). KEGG's own pathway hierarchy (`ko00001.keg`) is not
used -- it is licensed content -- so there are no pathway or category
totals; each KO links to its KEGG page on the site instead.

**Run order:** `Build_KEGG_DB.ipynb` first, then this notebook top to bottom.

In [1]:
import json
import time
from pathlib import Path
from collections import defaultdict
import sys
# The repo root: the nearest folder, from where this runs upward, that
# holds DataStructures/ -- so no path below is tied to one machine.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'DataStructures').is_dir())
sys.path.append(str(REPO / 'Notebooks'))
from All_Coverages import allfold
from weighted_totals import calculate_denominator_egc
from staleness_utils import find_stale_outputs


## 1. Load sample list and kegg_db


In [2]:
with open(REPO / 'sample_names.txt') as f:
    samples = [s.strip() for s in f]
print(f'Loaded {len(samples)} samples')

kegg_db_path    = REPO / 'DataStructures' / 'Kegg' / 'kegg_db.json'
metagenome_base = REPO / 'Metagenome_Samples'

# kegg_db.json is a prebuilt intermediate (produced by
# Build_KEGG_DB.ipynb from the raw .a.ko.txt files), unlike the other three
# EGC notebooks, which read their raw files directly every run. A stale
# intermediate would silently give stale KEGG EGC with no visible error --
# fail loudly here instead if any real input is newer than it.
#
# In practice this direction can't happen -- the raw JGI .a.ko.txt files
# are static 2023 downloads that never get touched again -- so this check
# alone doesn't guard the real risk: kegg_db.json getting rebuilt and this
# notebook's own .ko.json OUTPUTS never being regenerated to match. That
# complementary check (find_stale_outputs, Notebooks/staleness_utils.py)
# lives in the write cell below, since it's the write cell that knows both
# mtimes.
kegg_db_mtime = kegg_db_path.stat().st_mtime
stale_inputs = [
    s for s in samples
    if (metagenome_base / s / f'{s}.a.ko.txt').exists()
    and (metagenome_base / s / f'{s}.a.ko.txt').stat().st_mtime > kegg_db_mtime
]
if stale_inputs:
    raise RuntimeError(
        f"kegg_db.json is older than {len(stale_inputs)} of its real .a.ko.txt "
        f"inputs (e.g. {stale_inputs[0]}) -- re-run Build_KEGG_DB.ipynb before this notebook."
    )

with open(kegg_db_path) as f:
    kegg_db = json.load(f)
print(f'kegg_db: {len(kegg_db):,} genes')


Loaded 128 samples


kegg_db: 36,803,002 genes


## 1.2. Load KO names

`ko_names.json` (written by `Build_KEGG_DB.ipynb`) holds JGI's name for each
KO. A KO missing from it has no JGI name and is stored with `name = None`.

In [3]:
with open(kegg_db_path.parent / 'ko_names.json', encoding='utf-8') as f:
    ko_names = json.load(f)
print(f'KO names: {len(ko_names):,}')

KO names: 10,069


## 2. Build the KO list

One entry per KO. A gene counts once per KO per sample, even if it was
listed with the same KO more than once; its scaffold's gene count goes up by
one each time.

In [4]:
ko_tree = {}  # {ko_id: {name, timepoints}}

start = time.time()

for gene_id, ginfo in kegg_db.items():
    scaffold   = gene_id.rsplit('_', 2)[0]
    lookups    = ginfo.get('lookupId',  [])
    timepoints = ginfo.get('timepoint', [])

    for full_path, sample in zip(lookups, timepoints):
        # Exactly one id per entry, instead of silently taking the first of
        # a ';'-joined list -- this never fires on the real data, so if it
        # does, the raw .a.ko.txt format has changed.
        segments = full_path.split(';')
        assert len(segments) == 1, (
            f"Expected exactly one KEGG id per lookupId entry, got "
            f"{len(segments)} in {full_path!r} (gene {gene_id}) -- "
            f"the raw .a.ko.txt format may have changed."
        )
        ko_id = segments[0].strip()
        if ko_id.startswith('KO:'):
            ko_id = ko_id[3:]

        if ko_id not in ko_tree:
            ko_tree[ko_id] = {'name': ko_names.get(ko_id), 'timepoints': {}}

        node = ko_tree[ko_id]
        if sample not in node['timepoints']:
            node['timepoints'][sample] = {
                'egc':       0,
                'norm_egc':  0.0,
                'scaffolds': {},
                'genes':     set(),
            }

        tp = node['timepoints'][sample]
        if gene_id not in tp['genes']:
            tp['scaffolds'][scaffold] = tp['scaffolds'].get(scaffold, 0) + 1
        tp['genes'].add(gene_id)

named = sum(1 for n in ko_tree.values() if n['name'] is not None)
print(f'KOs: {len(ko_tree):,} ({named:,} with a JGI name)  ({time.time() - start:.1f}s)')

KOs: 12,286 (10,069 with a JGI name)  (69.3s)


## 3. Compute EGC and norm_egc

`egc = Σ (gene_count_on_scaffold × scaffold_coverage)`, using only genes annotated with a KEGG term (from this notebook's own `kegg_db`).

`norm_egc = egc / weighted_totals[sample]`, where the denominator comes from `Notebooks/weighted_totals.py` (`calculate_denominator_egc`) and is the SAME value used by `EGC_COG_TREE.ipynb` / `EGC_ENZYME_TREE.ipynb` / `EGC_TAX_TREE.ipynb` for a given sample -- computed once from every gene with any annotation across all four sources (via the mapping TSVs), not just genes with a KEGG hit. That shared scope is what makes norm_egc comparable across annotation types on the same plot. norm_egc still stays in (0, 1], since `egc` (a KEGG-only subset of the shared total) can never exceed it.

In [5]:
# Cross-source-comparable denominator.
#
# calculate_denominator_egc() (Notebooks/weighted_totals.py) computes ONE
# shared total per sample from the mapping TSVs -- every gene annotated by
# ANY of the four sources (taxonomy/KEGG/enzyme/COG), counted once each --
# so this notebook uses the exact same denominator as EGC_COG_TREE.ipynb /
# EGC_ENZYME_TREE.ipynb / EGC_TAX_TREE.ipynb for a given sample. That's what
# makes norm_egc comparable across annotation types on the same plot,
# instead of each notebook normalizing against only its own source's
# annotated genes.
weighted_totals = calculate_denominator_egc(samples)

# Sanity check on first sample: compare against the OLD unweighted,
# not-gene-count-weighted sum (allfold[sample]['total']) as a rough
# magnitude check -- not a correctness check on its own.
_s0 = next(iter(weighted_totals))
_new = weighted_totals[_s0]
_old = allfold.get(_s0, {}).get('total', 0)
print(f'Sample {_s0}: old denom={_old:,.0f}  new denom={_new:,.0f}')
print(f'norm_egc values will be ~{_old/_new:.2f}x smaller, now bounded by 1.0')

Sample 3300060358: old denom=8,298,642  new denom=16,137,978
norm_egc values will be ~0.51x smaller, now bounded by 1.0


In [6]:
def avg_gene_map(node_id):
    node = ko_tree[node_id]
    for sample_name, data in node['timepoints'].items():
        scaffold_counts = data.get('scaffolds')
        if not scaffold_counts or sample_name not in allfold:
            continue

        covs = allfold[sample_name]
        sample_total_depth = weighted_totals[sample_name]

        total_egc = 0
        for scaff, cnt in scaffold_counts.items():
            cov = covs.get(scaff, 0)
            total_egc += cnt * cov

        data['egc']      = total_egc
        data['norm_egc'] = (
            total_egc / sample_total_depth
            if sample_total_depth > 0 else 0.0
        )


for node_id in ko_tree:
    avg_gene_map(node_id)
print('EGC computation complete')


EGC computation complete


## 4. Write .ko.json files


In [7]:
base = REPO / 'DataStructures' / 'Kegg'
base.mkdir(parents=True, exist_ok=True)

# The real staleness risk isn't a raw .a.ko.txt being newer than
# kegg_db.json (cell 3's guard -- that direction can't actually happen,
# since the raw JGI files are static 2023 downloads) -- it's THESE outputs
# predating a kegg_db.json that's since been rebuilt. This doesn't block
# the write below -- writing fresh output IS the fix -- it just makes
# visible what this run is correcting, instead of silent.
existing_outputs = [base / f'{s}.ko.json' for s in samples]
stale_before_write = find_stale_outputs(existing_outputs, kegg_db_mtime)
if stale_before_write:
    print(
        f"NOTE: {len(stale_before_write)} of {len(existing_outputs)} existing "
        f".ko.json files were stale (older than kegg_db.json) before this run "
        f"-- refreshing all of them now."
    )

for s in samples:
    tp_out = {}
    for node_id, node in ko_tree.items():
        if s not in node['timepoints']:
            continue
        tp = node['timepoints'][s]
        tp_out[node_id] = {
            'name':        node['name'],
            'depth':       1,
            'parent_id':   None,
            'parent_name': None,
            'egc':         tp['egc'],
            'norm_egc':    tp['norm_egc'],
            'scaffolds':   tp.get('scaffolds', {}),
        }
    out_path = base / f'{s}.ko.json'
    with open(out_path, 'w') as f:
        json.dump(tp_out, f, indent=2)

print(f'Wrote {len(samples)} .ko.json files to {base}')


NOTE: 128 of 128 existing .ko.json files were stale (older than kegg_db.json) before this run -- refreshing all of them now.


Wrote 128 .ko.json files to c:\Users\sakoehn\Desktop\JGI\DataStructures\Kegg
